Name: Athena S. Villarin <br>
Name: Shane D. Canabo <br>
Year and Section: BSCS 3-A AI <br>
Date: February 12. 2026 <br>

### 1. Performing it manually

In [49]:
import re
from collections import Counter, defaultdict
import pandas as pd

# Dataset
data = [
    ("Free money now!!!", "SPAM"),
    ("Hi mom, how are you?", "HAM"),
    ("Lowest price for your meds", "SPAM"),
    ("Are we still on for dinner?", "HAM"),
    ("Win a free iPhone today", "SPAM"),
    ("Let's catch up tomorrow at the office", "HAM"),
    ("Meeting at 3 PM tomorrow", "HAM"),
    ("Get 50% off, limited time!", "SPAM"),
    ("Team meeting in the office", "HAM"),
    ("Click here for prizes!", "SPAM"),
    ("Can you send the report?", "HAM")
]

# Preprocessing function
def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)  # remove punctuation
    return text.split()

# Collect all words
all_words = set()
for doc, label in data:
    words = preprocess(doc)
    all_words.update(words)

vocabulary = sorted(list(all_words))
print("Vocabulary size:", len(vocabulary))

Vocabulary size: 44


In [50]:
# Display dataset in a table
dataset_df = pd.DataFrame(data, columns=['doc', 'class'])
print("Dataset:")
display(dataset_df)

Dataset:


,doc,class
0,Free money now!!!,SPAM
1,"Hi mom, how are you?",HAM
2,Lowest price for your meds,SPAM
3,Are we still on for dinner?,HAM
4,Win a free iPhone today,SPAM
5,Let's catch up tomorrow at the office,HAM
6,Meeting at 3 PM tomorrow,HAM
7,"Get 50% off, limited time!",SPAM
8,Team meeting in the office,HAM
9,Click here for prizes!,SPAM


In [51]:
# Calculate priors
total_docs = len(data)
ham_docs = sum(1 for _, label in data if label == "HAM")
spam_docs = sum(1 for _, label in data if label == "SPAM")

prior_ham = ham_docs / total_docs
prior_spam = spam_docs / total_docs

print(f"Total documents: {total_docs}")
print(f"HAM documents: {ham_docs}, Prior P(HAM): {prior_ham}")
print(f"SPAM documents: {spam_docs}, Prior P(SPAM): {prior_spam}")

Total documents: 11
HAM documents: 6, Prior P(HAM): 0.5454545454545454
SPAM documents: 5, Prior P(SPAM): 0.45454545454545453


In [52]:
# Calculate word counts per class
ham_word_counts = Counter()
spam_word_counts = Counter()

for doc, label in data:
    words = preprocess(doc)
    if label == "HAM":
        ham_word_counts.update(words)
    else:
        spam_word_counts.update(words)

total_ham_words = sum(ham_word_counts.values())
total_spam_words = sum(spam_word_counts.values())

# Display word counts in tables
ham_df = pd.DataFrame(list(ham_word_counts.items()), columns=['Word', 'HAM Count'])
spam_df = pd.DataFrame(list(spam_word_counts.items()), columns=['Word', 'SPAM Count'])

print("HAM Word Counts:")
display(ham_df)
print(f"Total HAM words: {total_ham_words}")

print("SPAM Word Counts:")
display(spam_df)
print(f"Total SPAM words: {total_spam_words}")

HAM Word Counts:


,Word,HAM Count
0,hi,1
1,mom,1
2,how,1
3,are,2
4,you,2
5,we,1
6,still,1
7,on,1
8,for,1
9,dinner,1


Total HAM words: 33
SPAM Word Counts:


,Word,SPAM Count
0,free,2
1,money,1
2,now,1
3,lowest,1
4,price,1
5,for,2
6,your,1
7,meds,1
8,win,1
9,a,1


Total SPAM words: 22


In [53]:
# Calculate likelihoods with Laplace smoothing
V = len(vocabulary)

ham_likelihoods = {}
spam_likelihoods = {}

for word in vocabulary:
    ham_likelihoods[word] = (ham_word_counts[word] + 1) / (total_ham_words + V)
    spam_likelihoods[word] = (spam_word_counts[word] + 1) / (total_spam_words + V)

# Display likelihoods in a table
likelihood_df = pd.DataFrame({
    'Word': vocabulary,
    'P(HAM|Word)': [ham_likelihoods[w] for w in vocabulary],
    'P(SPAM|Word)': [spam_likelihoods[w] for w in vocabulary]
})
print("Likelihoods (with Laplace smoothing):")
display(likelihood_df)

Likelihoods (with Laplace smoothing):


,Word,P(HAM|Word),P(SPAM|Word)
0,3,0.025974,0.015152
1,50,0.012987,0.030303
2,a,0.012987,0.030303
3,are,0.038961,0.015152
4,at,0.038961,0.015152
5,can,0.025974,0.015152
6,catch,0.025974,0.015152
7,click,0.012987,0.030303
8,dinner,0.025974,0.015152
9,for,0.025974,0.045455


In [54]:
# Function to classify a sentence
import math

def classify(sentence):
    words = preprocess(sentence)
    # Use log probabilities to avoid underflow
    ham_score = math.log(prior_ham)
    spam_score = math.log(prior_spam)
    
    for word in words:
        if word in ham_likelihoods:
            ham_score += math.log(ham_likelihoods[word])
            spam_score += math.log(spam_likelihoods[word])
        else:
            # If word not in vocab, use uniform probability
            ham_score += math.log(1 / (total_ham_words + V))
            spam_score += math.log(1 / (total_spam_words + V))
    
    print(f"Sentence: '{sentence}'")
    print(f"Words: {words}")
    print(f"HAM score: {ham_score:.4f}")
    print(f"SPAM score: {spam_score:.4f}")
    
    if ham_score > spam_score:
        return "HAM"
    else:
        return "SPAM"

In [55]:
# Test sentences
test1 = "Limited offer, click here!"
test2 = "Meeting at 2 PM with the manager"

print("Classification for test sentence 1:")
result1 = classify(test1)
print(f"Predicted class: {result1}\n")

print("Classification for test sentence 2:")
result2 = classify(test2)
print(f"Predicted class: {result2}")

Classification for test sentence 1:
Sentence: 'Limited offer, click here!'
Words: ['limited', 'offer', 'click', 'here']
HAM score: -17.9814
SPAM score: -15.4676
Predicted class: SPAM

Classification for test sentence 2:
Sentence: 'Meeting at 2 PM with the manager'
Words: ['meeting', 'at', '2', 'pm', 'with', 'the', 'manager']
HAM score: -26.7361
SPAM score: -30.1160
Predicted class: HAM


### 2. Using Scikit-Learn

In [56]:
# Using Scikit-Learn for Multinomial Naïve Bayes

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

# Dataset
data = [
    ("Free money now!!!", "SPAM"),
    ("Hi mom, how are you?", "HAM"),
    ("Lowest price for your meds", "SPAM"),
    ("Are we still on for dinner?", "HAM"),
    ("Win a free iPhone today", "SPAM"),
    ("Let's catch up tomorrow at the office", "HAM"),
    ("Meeting at 3 PM tomorrow", "HAM"),
    ("Get 50% off, limited time!", "SPAM"),
    ("Team meeting in the office", "HAM"),
    ("Click here for prizes!", "SPAM"),
    ("Can you send the report?", "HAM")
]

# Separate texts and labels
texts = [doc for doc, label in data]
labels = [label for doc, label in data]

# Create pipeline: vectorizer + classifier
pipeline = Pipeline([
    ('vectorizer', CountVectorizer(lowercase=True, token_pattern=r'\b\w+\b')),  # simple tokenization
    ('classifier', MultinomialNB())
])

# Train the model
pipeline.fit(texts, labels)

print("Model trained successfully.")

Model trained successfully.


In [57]:
# Test sentences
test_sentences = [
    "Limited offer, click here!",
    "Meeting at 2 PM with the manager"
]

# Predict classes
predictions = pipeline.predict(test_sentences)

for sentence, pred in zip(test_sentences, predictions):
    print(f"Sentence: '{sentence}'")
    print(f"Predicted class: {pred}")
    print()

# Also show probabilities
probabilities = pipeline.predict_proba(test_sentences)
classes = pipeline.classes_

for i, (sentence, probs) in enumerate(zip(test_sentences, probabilities)):
    print(f"Sentence: '{sentence}'")
    for cls, prob in zip(classes, probs):
        print(f"  P({cls}): {prob:.4f}")
    print()

Sentence: 'Limited offer, click here!'
Predicted class: SPAM

Sentence: 'Meeting at 2 PM with the manager'
Predicted class: HAM

Sentence: 'Limited offer, click here!'
  P(HAM): 0.0838
  P(SPAM): 0.9162

Sentence: 'Meeting at 2 PM with the manager'
  P(HAM): 0.9781
  P(SPAM): 0.0219

